# 2. Quantum kernel classification

This lab compares a classical radial-basis SVC with a quantum support vector classifier (QSVC). The quantum model uses exact statevector fidelities, so the experiment is deterministic and needs no cloud account.

## Learning goals

- prepare a nonlinear dataset without test-set leakage;
- encode two classical features in a `ZZFeatureMap`;
- inspect a fidelity kernel matrix; and
- compare quantum and classical models on the same split.

Install the project with `pip install -e ".[notebooks]"` before running this notebook.

In [ ]:
import numpy as np
from qiskit_machine_learning.kernels import FidelityStatevectorKernel

from qml_qiskit import build_feature_map, make_moons_split, run_benchmark

## Prepare the data

The training partition is scaled to rotation angles in `[0, π]`. The scaler never sees the test partition.

In [ ]:
data = make_moons_split(samples=60, noise=0.12, test_size=0.25, seed=42)
labels, counts = np.unique(data.train_labels, return_counts=True)

print("Training shape:", data.train_features.shape)
print("Test shape:", data.test_features.shape)
print("Training classes:", dict(zip(labels.tolist(), counts.tolist(), strict=True)))

## Encode features in a quantum circuit

The feature map converts each two-dimensional sample into a quantum state. Entangling gates let the resulting kernel represent interactions between features.

In [ ]:
feature_map = build_feature_map(data.num_features, reps=2)
print(feature_map.draw(output="text"))

## Inspect the fidelity kernel

Each matrix entry is the squared overlap between two encoded quantum states. A valid training kernel is symmetric, has ones on its diagonal, and is positive semidefinite up to numerical precision.

In [ ]:
kernel = FidelityStatevectorKernel(feature_map=feature_map, shots=None, enforce_psd=True)
kernel_matrix = kernel.evaluate(data.train_features)
eigenvalues = np.linalg.eigvalsh(kernel_matrix)

print("Kernel shape:", kernel_matrix.shape)
print("Smallest eigenvalue:", float(eigenvalues.min()))
print("First 5 x 5 block:\n", np.round(kernel_matrix[:5, :5], 3))

np.testing.assert_allclose(kernel_matrix, kernel_matrix.T, atol=1e-10)
np.testing.assert_allclose(np.diag(kernel_matrix), 1.0, atol=1e-10)
assert eigenvalues.min() > -1e-10

## Benchmark against a classical model

Both classifiers receive exactly the same training and test samples. Accuracy on a tiny synthetic split is illustrative, not evidence of practical quantum advantage.

In [ ]:
result = run_benchmark(data, seed=42, feature_map_reps=2)

print(f"Classical test accuracy: {result.classical.test_accuracy:.3f}")
print(f"Quantum test accuracy:   {result.quantum.test_accuracy:.3f}")
print(f"Quantum score delta:     {result.quantum_advantage:+.3f}")
result.as_dict()

## Next experiments

Change the seed, noise, sample count, or feature-map repetitions. Track both test accuracy and runtime: deeper feature maps can change the decision boundary, but they also raise simulation and hardware cost.